# 🎯 12 · Object Detection
### Finding every cell and drawing a box around it: sliding windows, anchors, and the one- vs. two-stage detector families

**Part F · Segmenting, detecting, synthesizing** · Module 12 of 29 (core) · builds on 08, 11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/12_object_detection.ipynb)

| | |
|---|---|
| **Hardware target** | CPU |
| **Core frameworks** | NumPy, SciPy |
| **Prerequisites** | Module 08 (CNN basics), Module 11 (deep segmentation) |
| **Estimated time** | 90–110 minutes |
| **02-740 topics covered** | sliding window + bounding-box regression, IoU, non-maximum suppression, YOLO, SSD, RetinaNet and focal loss, the R-CNN family (R-CNN, Fast/Faster R-CNN, RPN, ROI pooling/align, Mask R-CNN), detection applied to malaria parasite counting, combining watershed with a CNN classifier |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The detection image and its ground-truth boxes are generated by `course_toolkit.simulate_nuclei_image`, whose `boxes` output gives the exact (ymin, xmin, ymax, xmax) of every simulated cell.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain object detection as classification-plus-localization, and implement a from-scratch sliding-window
  detector with bounding-box scoring.
- Compute intersection-over-union (IoU) and use it both to evaluate detections and to perform
  non-maximum suppression (NMS).
- Explain, at the architecture level, how YOLO, SSD and RetinaNet turn detection into a single forward
  pass, and why focal loss addresses their foreground/background imbalance.
- Explain how the R-CNN family evolved (R-CNN → Fast R-CNN → Faster R-CNN → Mask R-CNN), including the
  region proposal network (RPN) and the difference between ROI pooling and ROI align.
- Describe how these ideas apply to a real bioimage task (malaria parasite counting) and how classical
  segmentation (watershed) can be combined with a CNN classifier as a lightweight detection pipeline.

## 🗺️ Contents

1. Detection as classification + localization: the sliding-window idea
2. Intersection-over-union (IoU): scoring a bounding box
3. Non-maximum suppression: from many candidate boxes to one per object
4. A from-scratch sliding-window detector, evaluated end to end
5. One-stage detectors: YOLO, SSD, RetinaNet and focal loss
6. Two-stage detectors: the R-CNN family
7. Application: malaria parasite counting, and combining watershed with a CNN

## 🧫 The biological question

Segmentation (Modules 10–11) answers "which pixels belong to an object," assuming we already know an
object is there. Many practical bioimage tasks instead start from "how many objects are in this field of
view, and where" — counting infected red blood cells on a malaria smear, counting colonies on a plate,
finding every cell nucleus in a tissue section before any downstream analysis happens. Object detection
answers exactly that question: find every instance, say where it is (a bounding box), and say what it is,
without necessarily tracing its exact pixel boundary.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 120
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def iou_boxes(box_a, box_b) -> float:
    """Intersection-over-union of two (ymin, xmin, ymax, xmax) boxes."""
    ya, xa, yA, xA = box_a
    yb, xb, yB, xB = box_b
    iy0, ix0 = max(ya, yb), max(xa, xb)
    iy1, ix1 = min(yA, yB), min(xA, xB)
    inter = max(0, iy1 - iy0) * max(0, ix1 - ix0)
    area_a = max(0, yA - ya) * max(0, xA - xa)
    area_b = max(0, yB - yb) * max(0, xB - xb)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def non_max_suppression_boxes(boxes: np.ndarray, scores: np.ndarray, iou_threshold: float = 0.3):
    """Greedy non-maximum suppression: keep the highest-scoring box, remove boxes that overlap
    it by more than ``iou_threshold``, repeat. Returns the kept indices (into ``boxes``)."""
    order = np.argsort(-scores)
    keep = []
    while len(order):
        i = order[0]
        keep.append(i)
        rest = order[1:]
        order = np.array([j for j in rest if iou_boxes(boxes[i], boxes[j]) <= iou_threshold])
    return keep


def precision_recall_f1(y_true: np.ndarray, y_pred: np.ndarray):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 for binary labels
    (1 = positive). Returns (precision, recall, f1)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


def overlay_boxes(ax, boxes, color: str = "lime", linewidth: float = 1.5):
    """Draw (ymin, xmin, ymax, xmax) boxes as rectangles on an existing image axis."""
    import matplotlib.patches as patches
    for ymin, xmin, ymax, xmax in boxes:
        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                  linewidth=linewidth, edgecolor=color, facecolor="none")
        ax.add_patch(rect)


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

---

## 1. Detection as classification + localization: the sliding-window idea

The most direct way to turn a classifier into a detector: slide a fixed-size window across the image at
every position (and, for objects of unknown size, at several scales), run the classifier on each window's
contents, and keep the positions where it fires "object" with high confidence. This immediately raises two
problems this notebook works through in order: a fixed-size window rarely matches an object's actual
extent exactly, so a detector typically also predicts a **bounding-box regression** — four numbers nudging
the window's position and size to better fit the object it found — and, because nearby windows overlap
heavily, one true object triggers *many* nearby high-confidence windows, which Section 3's non-maximum
suppression must collapse back down to one detection per object.

In [ ]:
# ── 1.1 Visualize the sliding-window scan over a field of simulated cells ──
scene = simulate_nuclei_image(n_cells=10, size=96, touching=False, noise_sigma=0.03, seed=21)
image, true_boxes = scene["image"], scene["boxes"]

window_size, stride = 18, 9
window_centers = [(r, c) for r in range(window_size // 2, image.shape[0] - window_size // 2, stride)
                          for c in range(window_size // 2, image.shape[1] - window_size // 2, stride)]
print(f"{len(window_centers)} candidate window positions for a {image.shape} image "
      f"({window_size}x{window_size} windows, stride {stride})")

fig, ax = plt.subplots(figsize=(5, 5))
show_image(ax, image, f"{len(true_boxes)} true cells + a sample of scan windows")
overlay_boxes(ax, true_boxes, color="lime")
sample = window_centers[::7]
sample_boxes = np.array([[r - window_size // 2, c - window_size // 2,
                           r + window_size // 2, c + window_size // 2] for r, c in sample])
overlay_boxes(ax, sample_boxes, color="yellow")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Green boxes are the ground truth (known exactly,
because the simulator placed every cell itself); yellow boxes are a *sample* of the sliding-window scan
positions (plotting all of them would be an unreadable grid). Even at this modest stride, several yellow
windows already sit close to each green box — a real sliding-window scan at full density produces many
more candidates than that, one reason Sections 2–3 are needed before this becomes a usable detector.

## 2. Intersection-over-union (IoU): scoring a bounding box

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

IoU is the standard way to compare two boxes: 0 when they don't overlap at all, 1 when they coincide
exactly. It plays two different roles in detection: as an **evaluation metric** (is a predicted box "close
enough" to a true box — conventionally IoU $\geq 0.5$ counts as a correct detection) and, in Section 3, as
the criterion non-maximum suppression uses to decide that two candidate boxes are "the same detection.

In [ ]:
# ── 2.1 IoU between a sliding-window box and the true boxes it overlaps ──
probe_box = np.array([20, 20, 38, 38])   # (ymin, xmin, ymax, xmax)
ious_to_truth = [iou_boxes(probe_box, tb) for tb in true_boxes]
best_match = int(np.argmax(ious_to_truth))
print("IoU of the probe window against every true box:", np.round(ious_to_truth, 3))
print(f"best match: true box #{best_match}, IoU={ious_to_truth[best_match]:.3f} "
      f"({'counts' if ious_to_truth[best_match] >= 0.5 else 'does NOT count'} as a correct detection "
      "at the conventional 0.5 threshold)")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
show_image(ax, image, "probe window (yellow) vs. true boxes (green)")
overlay_boxes(ax, true_boxes, color="lime")
overlay_boxes(ax, probe_box[None, :], color="yellow")
plt.tight_layout(); plt.show()

✏️ **Your turn.** Change `probe_box` in Section 2.1 to shift it a further 10
pixels in `x` (e.g. `np.array([20, 30, 38, 48])`) and re-run. At what point does the best-matching IoU drop
below 0.5? This is exactly the quantity a bounding-box regressor is trained to improve: starting from a
coarse window like this one, predict a small shift and resize that increases IoU against the true box.

## 3. Non-maximum suppression: from many candidate boxes to one per object

A real sliding-window (or anchor-based) detector produces a *score* for every candidate box — the
classifier's confidence that this window contains an object. Near any true object, many overlapping
candidate boxes all score highly. **Non-maximum suppression (NMS)** is the standard greedy post-processing
step that collapses these duplicates: repeatedly take the highest-scoring remaining box, keep it, and
discard every other remaining box whose IoU against it exceeds a threshold (meaning "this is probably the
same object"), until no candidates remain.

In [ ]:
# ── 3.1 Non-maximum suppression on a cluster of overlapping candidate boxes ──
cluster_boxes = np.array([
    [20, 20, 38, 38], [21, 23, 39, 41], [19, 19, 36, 37], [22, 25, 40, 43],   # cluster around one object
    [60, 60, 78, 78], [61, 62, 80, 80],                                       # cluster around a 2nd object
])
cluster_scores = np.array([0.91, 0.95, 0.80, 0.70, 0.88, 0.60])

keep_idx = non_max_suppression_boxes(cluster_boxes, cluster_scores, iou_threshold=0.3)
print(f"{len(cluster_boxes)} candidate boxes -> {len(keep_idx)} kept after NMS: indices {keep_idx}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4.3))
for ax, boxes, title in [(axes[0], cluster_boxes, "before NMS (all candidates)"),
                          (axes[1], cluster_boxes[keep_idx], "after NMS (one per object)")]:
    ax.set_xlim(0, 100); ax.set_ylim(100, 0); ax.set_aspect("equal"); ax.set_title(title)
    overlay_boxes(ax, boxes, color="orange")
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** NMS kept exactly the single *highest-scoring* box
from each cluster (index 1, score 0.95, from the first cluster; index 4, score 0.88, from the second) —
confirm this matches `keep_idx` above. Lowering `iou_threshold` makes NMS more aggressive (suppressing
boxes that overlap even a little); raising it toward 1.0 would let near-duplicate boxes survive.

## 4. A from-scratch sliding-window detector, evaluated end to end

We now assemble Sections 1–3 into a complete (if deliberately simple) detector: score every sliding-window
position with a classical **matched filter** (correlation against a Gaussian "cell template" — no training
required, in the spirit of Module 10's classical methods, so the detection pipeline itself is the focus
here rather than the scoring function), threshold to get candidate boxes, run NMS, and evaluate the
survivors against the ground-truth boxes using IoU-based matching.

In [ ]:
# ── 4.1 Score every window with a Gaussian matched filter, then threshold ──
def gaussian_template(size, sigma):
    ax = np.arange(size) - size // 2
    xx, yy = np.meshgrid(ax, ax)
    t = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return (t - t.mean()) / t.std()

template = gaussian_template(window_size, sigma=4.0)
response = ndi.correlate(image - image.mean(), template, mode="constant")

candidate_boxes, candidate_scores = [], []
for r, c in window_centers:
    score = response[r, c]
    candidate_boxes.append([r - window_size // 2, c - window_size // 2,
                             r + window_size // 2, c + window_size // 2])
    candidate_scores.append(score)
candidate_boxes = np.array(candidate_boxes)
candidate_scores = np.array(candidate_scores)

SCORE_THRESHOLD = np.percentile(candidate_scores, 90)   # keep the top 10% of windows as candidates
above = candidate_scores >= SCORE_THRESHOLD
print(f"{above.sum()} / {len(candidate_boxes)} windows pass the score threshold "
      f"({SCORE_THRESHOLD:.2f})")

In [ ]:
# ── 4.2 Non-maximum suppression, then evaluate against the ground truth ──
kept = non_max_suppression_boxes(candidate_boxes[above], candidate_scores[above], iou_threshold=0.2)
final_boxes = candidate_boxes[above][kept]
final_scores = candidate_scores[above][kept]
print(f"{len(final_boxes)} final detections after NMS (true cell count: {len(true_boxes)})")

# greedy IoU matching: each predicted box claims its best unclaimed true box, at IoU >= 0.5
IOU_MATCH = 0.5
claimed = set()
true_positive = np.zeros(len(final_boxes), dtype=bool)
for i in np.argsort(-final_scores):
    ious = [iou_boxes(final_boxes[i], tb) if j not in claimed else -1 for j, tb in enumerate(true_boxes)]
    best_j = int(np.argmax(ious))
    if ious[best_j] >= IOU_MATCH:
        true_positive[i] = True
        claimed.add(best_j)

y_pred_all = np.ones(len(final_boxes))        # every surviving detection is a positive prediction
y_true_all = true_positive.astype(int)         # 1 = matched a real cell, 0 = a false positive
precision = true_positive.sum() / len(final_boxes)
recall = len(claimed) / len(true_boxes)
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
print(f"precision={precision:.2f}  recall={recall:.2f}  F1={f1:.2f}  "
      f"({len(claimed)}/{len(true_boxes)} true cells matched)")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
show_image(ax, image, f"detections: green=true, lime-dashed=correct, red=false positive")
overlay_boxes(ax, true_boxes, color="lime")
if true_positive.any():
    overlay_boxes(ax, final_boxes[true_positive], color="cyan")
if (~true_positive).any():
    overlay_boxes(ax, final_boxes[~true_positive], color="red")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Cyan boxes are detections that matched a true cell
at IoU $\geq 0.5$ (true positives); red boxes are detections with no good match (false positives); any
green box with no cyan box on it is a missed cell (false negative). This whole pipeline — score candidate
windows, threshold, NMS, IoU-match against ground truth — is exactly the evaluation protocol used for every
detector in Sections 5–6, whether the scoring function is a hand-built matched filter (as here) or a
trained deep network.

✏️ **Your turn.** Lower `SCORE_THRESHOLD`'s percentile from 90 to 75 in Section
4.1 and re-run Sections 4.1–4.2. You should see recall increase (more true cells get matched) at the cost
of precision (more false-positive boxes survive) — the same precision/recall trade-off that a detector's
confidence threshold controls in every architecture below.

### 📘 Lecture notes — why a hand-built scoring function, here

Real detectors replace the matched-filter score in Section 4.1 with a trained classifier's confidence —
Module 08's `SmallCNN` applied to each window's contents would be a direct drop-in replacement, scored once
per window exactly as the matched filter was here. The sliding-window-plus-NMS *pipeline* is identical
either way; only the per-window scoring function changes from classical (fast, no training, limited to
simple/uniform-looking objects) to learned (slower to develop, far more general).

## 5. One-stage detectors: YOLO, SSD, RetinaNet and focal loss

Scoring thousands of independent sliding windows, as Section 4 did, is wasteful: a CNN's convolutional
layers already compute features at every spatial location in one shared pass. **One-stage detectors**
exploit this: a single forward pass through one network directly produces, at every location of a coarse
output grid, both a confidence score *and* bounding-box coordinates for one or more candidate boxes
("anchors") of different predefined sizes/aspect ratios centered there — classification and localization,
together, in one shot, with no separate second pass.

- **YOLO** ("You Only Look Once") divides the image into a coarse grid; each grid cell directly predicts a
  fixed number of boxes (offsets from an anchor), a confidence, and class probabilities. Extremely fast,
  historically at some cost to localizing small or crowded objects precisely.
- **SSD** (Single Shot Detector) predicts from *several* feature-map resolutions simultaneously (not just
  one, like the original YOLO), so different layers naturally specialize in detecting objects of different
  sizes — large objects from coarse/deep layers, small objects from fine/shallow layers.
- **RetinaNet** is also one-stage and multi-scale (like SSD), but its key contribution is **focal loss**:

$$\text{FL}(p_t) = -(1-p_t)^\gamma \log(p_t)$$

  Most anchor locations in any image are easy background, so plain cross-entropy's loss is dominated by a
  huge number of easy-negative anchors, drowning out the signal from the rare, hard, truly-informative
  ones. The $(1-p_t)^\gamma$ factor **down-weights already-easy, correctly-classified examples**
  ($p_t \to 1 \Rightarrow (1-p_t)^\gamma \to 0$), letting the loss focus on the hard examples that
  still need to improve — directly solving the foreground/background imbalance that motivated two-stage
  detectors' separate proposal stage in the first place (Section 6), without needing one.

In [ ]:
# ── 5.1 Focal loss down-weights easy examples relative to cross-entropy ──
p_t = np.linspace(0.01, 0.99, 200)
ce_loss = -np.log(p_t)
for gamma in [0, 1, 2, 5]:
    fl = -(1 - p_t) ** gamma * np.log(p_t)
    plt.plot(p_t, fl, label=f"focal loss, γ={gamma}" + ("  (=cross-entropy)" if gamma == 0 else ""))
plt.xlabel("$p_t$ (predicted probability of the true class)"); plt.ylabel("loss")
plt.title("Focal loss vs. plain cross-entropy (γ=0)"); plt.legend(); plt.tight_layout(); plt.show()

print(f"at p_t=0.9 (an easy, already-confident example): "
      f"CE={-(np.log(0.9)):.3f}, focal(γ=2)={-(1-0.9)**2*np.log(0.9):.4f}  "
      f"-> focal loss is {(-(np.log(0.9)))/(-(1-0.9)**2*np.log(0.9)):.0f}x smaller")

🔍 **What just happened?** At $\gamma=0$, focal loss *is* cross-entropy. As
$\gamma$ grows, the curve flattens out near $p_t=1$ (easy examples contribute almost nothing) while
staying steep near $p_t=0$ (hard, wrong examples still contribute a large loss) — exactly the reweighting
that lets a one-stage detector train stably even when 99%+ of its anchor locations are easy background.

## 6. Two-stage detectors: the R-CNN family

**Two-stage detectors** split the job Section 5's networks do in one pass into two: first *propose* a
modest number of candidate regions likely to contain *some* object (without yet saying which class), then
*classify and refine* only those proposals. This trades speed for (historically) higher accuracy,
especially on small or unusual objects.

- **R-CNN** (2014): an external algorithm (selective search) proposes ~2000 regions; each is warped to a
  fixed size and run independently through a CNN, then classified by an SVM. Accurate but extremely slow —
  the CNN runs once *per proposal*.
- **Fast R-CNN** (2015): run the CNN *once* over the whole image, then use **ROI pooling** to extract a
  fixed-size feature-map crop for each proposal directly from the shared feature map (instead of
  reprocessing each proposal from pixels) — a large speedup, and classification + bounding-box regression
  are trained jointly, end to end.
- **Faster R-CNN** (2015): replaces the external region-proposal algorithm with a learned **region
  proposal network (RPN)** — a small CNN head, sharing the same backbone features, that itself predicts
  objectness scores and box offsets for a dense grid of anchors, producing proposals as part of the same
  network rather than a separate, non-learned preprocessing step.
- **Mask R-CNN** (2017): adds a third output branch to Faster R-CNN — alongside class and box, a small
  per-proposal segmentation mask (a mini U-Net-style decoder on each ROI's cropped features) — turning
  object detection directly into *instance segmentation*. It also replaces ROI pooling with **ROI align**,
  which uses bilinear interpolation instead of pooling's integer-coordinate rounding, avoiding small
  misalignments between a proposal's true boundary and its extracted features — important once the output
  is a pixel-precise mask rather than just a box.

In [ ]:
# ── 6.1 ROI pooling vs. ROI align: why rounding hurts small objects ──
feature_map = np.arange(8 * 8).reshape(8, 8).astype(float)   # a small, schematic feature map
roi = (1.4, 1.6, 5.4, 5.6)   # a proposal's (ymin, xmin, ymax, xmax) in feature-map coordinates

# ROI pooling (classic): round the ROI boundary to the nearest integer grid cell before pooling
r0, c0, r1, c1 = [int(round(v)) for v in roi]
pooled_region = feature_map[r0:r1, c0:c1]
print("ROI pooling rounds the boundary to:", (r0, c0, r1, c1), "-> pooled mean:", pooled_region.mean().round(2))

# ROI align (conceptual): sample at the *exact* (fractional) boundary via bilinear interpolation
from scipy.ndimage import map_coordinates
yy, xx = np.meshgrid(np.linspace(roi[0], roi[2], 4), np.linspace(roi[1], roi[3], 4), indexing="ij")
aligned_samples = map_coordinates(feature_map, [yy.ravel(), xx.ravel()], order=1).reshape(4, 4)
print("ROI align samples the exact boundary", roi, "-> pooled mean:", aligned_samples.mean().round(2))

🔍 **What just happened?** ROI pooling's integer rounding moved the proposal's
true boundary (1.4, 1.6, 5.4, 5.6) to (1, 2, 5, 6) before extracting anything — a fixed, systematic
misalignment of up to half a feature-map cell on every side. ROI align instead samples exactly at the
true fractional boundary via bilinear interpolation, so the extracted features line up with the actual
proposal regardless of its position — the difference that matters once the downstream task (a per-pixel
mask, in Mask R-CNN) is sensitive to exactly where that boundary falls.

✏️ **Your turn.** Shrink the ROI in Section 6.1 to `roi = (1.4, 1.6, 2.6,
2.8)` (a much smaller proposal, more typical of a small cell in a crowded field) and re-run. Rounding error
of the same absolute size is a much larger *fraction* of a small ROI than a large one — which is exactly
why ROI align's benefit over ROI pooling is most pronounced for small objects, a common case in bioimage
detection.

## 7. Application: malaria parasite counting, and combining watershed with a CNN

A concrete, widely used bioimage application: detecting and counting malaria parasites in stained blood
smear images. Infected red blood cells must be found among a large field of uninfected ones — exactly the
sliding-window-plus-classifier (or one-stage detector) setup of Sections 4–5, with the practical wrinkle
that microscopy fields contain *hundreds* of cells per image, making detection speed and the
false-positive rate (Section 5's imbalance problem) especially important: a detector that is only 99.9%
specific still produces one false positive for every thousand uninfected cells scanned, which can swamp
the true parasite count in a lightly-infected sample.

A practical, lower-engineering-cost alternative when segmentation is already reliable (Module 10):
**combine classical segmentation with a CNN classifier**, rather than training a full detector end to end.
Marker-controlled watershed (or any other classical/learned segmentation) first separates the image into
individual cell candidates, each cell's cropped sub-image is then classified independently (Module 08's
`SmallCNN`, or Module 07's classical features, applied per-crop) as infected/uninfected or by cell type.
This sidesteps learning bounding-box regression and anchor matching entirely — segmentation already solved
the localization problem — at the cost of being only as good as the upstream segmentation, and of treating
touching/overlapping objects exactly as fragile as Module 11's weighted loss was designed to address.

In [ ]:
# ── 7.1 Schematic: the "segment then classify" detection pipeline ──
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
demo = simulate_nuclei_image(n_cells=9, size=72, touching=False, seed=42)
show_image(axes[0], demo["image"], "1. input field")
axes[1].imshow(demo["instance_mask"], cmap="nipy_spectral"); axes[1].set_title("2. watershed segmentation")
axes[1].set_xticks([]); axes[1].set_yticks([])
show_image(axes[2], demo["image"], "3. classify each candidate crop")
overlay_boxes(axes[2], demo["boxes"], color="yellow")
for i, (ymin, xmin, ymax, xmax) in enumerate(demo["boxes"]):
    label = "infected" if demo["shape_labels"][i] in ("irregular", "ring") else "uninfected"
    axes[2].text(xmin, ymin - 2, label, color="yellow", fontsize=6)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** This schematic relabels two of the six simulated
shape classes ("irregular", "ring") as a stand-in "infected" category purely to illustrate the pipeline —
confirm that the labels in the rightmost panel line up with exactly those two shape classes among the nine
simulated cells, and not the other four ("round", "dividing", "elongated", "kidney").

📌 **Key takeaways**

- Object detection is classification *and* localization together: a bounding box plus a class/confidence for every detected instance.
- IoU is the common currency of detection: it defines both when two candidate boxes count as 'the same object' (non-maximum suppression) and when a predicted box counts as a correct match to a ground-truth box (evaluation).
- One-stage detectors (YOLO, SSD, RetinaNet) predict boxes and classes in a single forward pass over a dense anchor grid; focal loss specifically counters the resulting foreground/background imbalance by down-weighting easy examples.
- Two-stage detectors (the R-CNN family) first propose candidate regions, then classify and refine only those; Faster R-CNN's RPN makes proposal generation itself learned, and Mask R-CNN extends the idea to per-proposal instance masks using ROI align.
- When reliable segmentation already exists, 'segment then classify each crop' is a practical detection pipeline that avoids training box regression and anchor matching outright.

✅ **Checkpoint**

1. In Section 4, why was non-maximum suppression necessary even after thresholding out low-scoring
   windows?
   <details><summary>Answer</summary>Because many *overlapping* windows near the same true object all
   score above the threshold — thresholding alone removes low-confidence noise but does nothing about
   duplicate high-confidence detections of the same object, which NMS collapses to one.</details>
2. Why does focal loss reduce to ordinary cross-entropy when $\gamma = 0$, and what does increasing
   $\gamma$ change?
   <details><summary>Answer</summary>$(1-p_t)^0 = 1$ for any $p_t$, so the reweighting factor disappears
   and FL becomes exactly $-\log(p_t)$ (cross-entropy). Increasing $\gamma$ makes the down-weighting of
   easy, high-$p_t$ examples more aggressive, concentrating the loss on hard examples.</details>
3. What specific problem does Faster R-CNN's RPN solve relative to R-CNN and Fast R-CNN?
   <details><summary>Answer</summary>Both R-CNN and Fast R-CNN depend on an external, non-learned region
   proposal algorithm (e.g. selective search), which is slow and not optimized jointly with the rest of the
   network. The RPN produces proposals as a learned part of the same network, sharing the backbone's
   features, making the whole pipeline end-to-end trainable and much faster.</details>

🧠 **Questions & discussion**

- Section 7's 'segment then classify' pipeline sidesteps learning bounding boxes entirely. What specifically would break this pipeline that a true end-to-end detector would handle more gracefully?
- One-stage detectors predict boxes relative to predefined anchors of fixed sizes/aspect ratios. What would happen to detection quality if the true objects' sizes varied *much* more than the anchor set was designed for (e.g. parasites ranging from 1 to 50 pixels in diameter in the same image)?
- Malaria smear images can contain hundreds of red blood cells, only a few of which are infected. Given Section 5's focal-loss discussion, why is this specific imbalance ratio especially demanding for a detector, and not just mildly inconvenient?

🚀 **Bonus — going further**

- Extend Section 4's matched-filter detector to *multiple scales* (try `window_size` values of 14, 18, and 24 and pool all candidates together before NMS) and see whether recall improves on cells of varying size.
- Implement a simple bounding-box regression: for each ground-truth cell, record the offset from its nearest anchor center, fit a linear regression from local image features to that offset, and check whether it improves the IoU of the best-matching candidate box.
- Combine Module 08's `SmallCNN` as the per-window scorer in Section 4's pipeline (replacing the matched filter) and compare precision/recall to the classical scoring function.

📝 **Lab note**

```text
Module 12 — Object detection
Date: ____________
Sliding-window detector: precision=____  recall=____  F1=____
Score threshold used (percentile): ____   NMS IoU threshold used: ____
One false positive and why I think it happened: ____________
One missed true cell and why I think it happened: ____________
```

📚 **References**

- Girshick, R., Donahue, J., Darrell, T. & Malik, J. (2014). Rich feature hierarchies for accurate object detection and semantic segmentation (R-CNN). *CVPR 2014*.
- Girshick, R. (2015). Fast R-CNN. *ICCV 2015*.
- Ren, S., He, K., Girshick, R. & Sun, J. (2015). Faster R-CNN: Towards real-time object detection with region proposal networks. *NeurIPS 2015*.
- He, K., Gkioxari, G., Dollár, P. & Girshick, R. (2017). Mask R-CNN. *ICCV 2017*.
- Redmon, J., Divvala, S., Girshick, R. & Farhadi, A. (2016). You Only Look Once: Unified, real-time object detection (YOLO). *CVPR 2016*.
- Liu, W. et al. (2016). SSD: Single shot multibox detector. *ECCV 2016*.
- Lin, T.-Y., Goyal, P., Girshick, R., He, K. & Dollár, P. (2017). Focal loss for dense object detection (RetinaNet). *ICCV 2017*.
- Rajaraman, S. et al. (2018). Pre-trained convolutional neural networks as feature extractors toward improved malaria parasite detection in thin blood smear images. *PeerJ*, 6, e4568.

➡️ **Next.** Module 13 turns from recognizing and locating existing structures to *generating* new, realistic ones: shape-space models, variational autoencoders, and generative adversarial networks applied to synthetic and real bioimages.